In [4]:
from bronze_ingest.archive import cleanup_old_raw_files
from bronze_ingest.file_discovery import (
    get_s3_bucket_contents,
    get_latest_dump_date_from_metrics_uc,
    get_latest_dump_files_info_s3,
)
from unitycatalog.uc import uc_create_catalog, uc_list_tables, uc_create_schemas
from download_raw.validate import validate_downloads_s3


from settings import Settings
import logging
from pathlib import Path


from logs.logging_config import setup_logger
from helpers_spark.delta_metrics import (
    create_new_delta_history,
    export_delta_table_history,
)
from helpers_spark.schemas import export_schemas
from helpers_spark.spark_session import create_spark_session
from bronze_ingest.xml import read_xml_to_df
from helpers_spark.hash import apply_hash
from helpers_spark.delta_tables import (
    merge_into_delta,
    create_new_delta_table_s3,
    infer_columns_from_delta,
)
from helpers_spark.serialization import serialize_for_merge
from unitycatalog.uc import uc_register_table

import sys

import datetime


ModuleNotFoundError: No module named 'app'

In [ ]:
def create_and_uc_register_new_delta_table(
    df, catalog, schema, table_name, output_dir, HEADERS, UC_URL, logger
):

    output_table_s3_path = output_dir + "/"+table_name

    uc_create_schemas(
        catalog, schema_list=[schema], HEADERS=HEADERS, UC_URL=UC_URL, logger=logger
    )


    create_new_delta_table_s3(
        df,
        output_table_s3_path,
        logger=logger,
    )
    columns = infer_columns_from_delta(output_table_s3_path, spark)

    uc_register_table(
        catalog, schema, table_name, output_dir, columns, HEADERS, UC_URL, logger
    )



In [ ]:
def process_single_dump(spark, dump, settings, logger, catalog, s3, schema="raw"):

    config_bronze = settings.bronze

    dump_type = dump["dump_type"]

    if dump_type not in config_bronze["PRIMARY_KEYS"]:
        return None

    primary_key = config_bronze["PRIMARY_KEYS"][dump_type]

    latest_dump_date_to_process = dump["latest_dump_date"]
    
    
    input_file_path = f's3a://{storage.bucket}/{dump["file"]}'

    logger.info(f"Processing {input_file_path}")

    logger.info(f"{config_bronze = }")

    HEADERS = settings.uc.headers
    UC_URL = settings.uc.url
    metrics_schema = "metrics"
    table_name = "bronze"  # table_name in UC

    latest_recorded_dump_date = get_latest_dump_date_from_metrics_uc(
        dump_type, catalog, metrics_schema, table_name, HEADERS, UC_URL, spark, logger
    )

    logger.info(f"{latest_recorded_dump_date = }")

    try:
        if latest_recorded_dump_date:
            prev = datetime.datetime.strptime(latest_recorded_dump_date, "%Y%m%d")
            curr = datetime.datetime.strptime(latest_dump_date_to_process, "%Y%m%d")

            if prev >= curr:
                logger.info(
                    "Skipping dump %s (%s) because previous dump %s already processed",
                    dump_type,
                    latest_dump_date_to_process,
                    latest_recorded_dump_date,
                )
                return None

        start = datetime.datetime.now()
        df = read_xml_to_df(spark, dump_type, input_file_path, logger=logger)

        df_hashed = apply_hash(
            df,
            primary_key=primary_key,
            latest_dump_date=latest_dump_date_to_process,
            hash_col="root_hash",
        )

        if config_bronze["COLS_TO_HASH"][dump_type]:
            for col in config_bronze["COLS_TO_HASH"][dump_type]:
                df_hashed = apply_hash(
                    df_hashed,
                    primary_key=primary_key,
                    latest_dump_date=latest_dump_date_to_process,
                    hash_col=f"{col}_hash",
                    col_to_hash=col,
                )

        df_mergeable = serialize_for_merge(df_hashed, primary_key=primary_key)

        output_dir = f"s3a://{storage.bucket}/{catalog}"

        if dump_type not in uc_list_tables(
            catalog, schema, HEADERS, UC_URL, logger=logger
        ):
            create_and_uc_register_new_delta_table(
                df_mergeable,
                catalog,
                schema,
                table_name,
                output_dir,
                HEADERS,
                UC_URL,
                logger,
            )

        else:
            merge_into_delta(
                spark,
                df_mergeable,
                output_dir,
                primary_key,
                hash_col="root_hash",
                logger=logger,
            )

        export_schemas(
            df_mergeable,
            dump_type=dump_type,
            dump_date=latest_dump_date_to_process,
            logger=logger,
            metadata_dir=settings.metadata_dir,
        )
        export_delta_table_history(
            spark,
            dump_type,
            latest_dump_date_to_process,
            output_dir=output_dir,
            input_layer="raw",
            output_layer="bronze",
            logger=logger,
            LOG_DIR=settings.log_dir,
            table_name=f"bronze_{dump_type}_metrics"
        )

        # cleanup_old_raw_files(
        #    latest_dump_date_to_process, dump_type, settings.raw_data_dir
        # )

        duration = (datetime.datetime.now() - start).total_seconds()
        logger.info(
            "dump_processed",
            extra={
                "dump_date": latest_dump_date_to_process,
                "dump_type": dump_type,
                "duration": duration,
            },
        )

        return {
            "dump_type": dump_type,
            "dump_date": latest_dump_date_to_process,
            "status": "processed",
            "duration": duration,
        }

    except Exception:
        logger.exception(
            "Failed processing dump %s-%s",
            latest_dump_date_to_process,
            dump_type,
        )
        raise


In [ ]:
def main(spark, settings, bucket: str, logger, s3):
    HEADERS = settings.uc.headers
    UC_URL = settings.uc.url
    catalog = settings.project_name + "_" + settings.storage.env
    
    uc_create_catalog(catalog, HEADERS=HEADERS, UC_URL=UC_URL, logger=logger)
    
    uc_create_schemas(
        catalog, schema_list=["metrics","raw"], HEADERS=HEADERS, UC_URL=UC_URL, logger=logger
    )

    if settings.storage.type == "minio":
        schema = "raw"

        validate_downloads_s3(
            bucket=settings.project_name,
            catalog=catalog,
            schema=schema,
            logger=logger,
            s3=s3,
            dump_types_to_process=settings.raw["DUMP_TYPE_TO_PROCESS"][settings.env],
        )

        raw_dump_list = get_s3_bucket_contents(
            bucket=bucket, catalog=catalog, schema=schema, logger=logger, s3=s3
        )

        logger.info(f"{raw_dump_list = }")

        raw_dumps = get_latest_dump_files_info_s3(raw_dump_list, logger)
        logger.info(f"{raw_dumps = }")



        results = []
        for dump in raw_dumps:
            result = process_single_dump(spark, dump, settings, logger, catalog, s3)
            if result:
                results.append(result)


In [ ]:
    settings = Settings.load()
    storage = settings.storage

    try:
        s3 = storage.s3

        spark = create_spark_session(
            app_name="discogs-bronze-ingest",
            endpoint=storage.endpoint,
            access_key=storage.access_key,
            secret_key=storage.secret_key,
        )

        setup_logger(
            settings.log_dir
            / f"{settings.project_name}_{settings.env}_bronze_ingest_log.json",
            settings.env,
            settings.project_name,
            storage_type=settings.storage.type,
        )

        logger = logging.getLogger(Path(__file__).stem)

        try:
            main(
                spark,
                settings,
                bucket=settings.project_name,
                logger=logger,
                s3=s3,
            )
        finally:
            spark.stop()
    except Exception:
        logger.exception("Fatal error during Bronze ingest job")
        sys.exit(1)
